In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab and Kaggle notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29 peft trl triton
    !pip install --no-deps cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    "unsloth/llama-2-7b-bnb-4bit",
    "unsloth/llama-2-13b-bnb-4bit",
    "unsloth/codellama-34b-bnb-4bit",
    "unsloth/tinyllama-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit", # New Google 6 trillion tokens model 2.5x faster!
    "unsloth/gemma-2b-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit", # Choose ANY! eg teknium/OpenHermes-2.5-Mistral-7B
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.3.3: Fast Mistral patching. Transformers: 4.48.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.3.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
#df=pd.read_json('/content/sample_data/train_data.json')

In [ ]:
import pandas as pd
from datasets import Dataset
import json
import pandas as pd

# Read JSON file line by line and append each JSON object to a list
data = []
with open('/content/sample_data/train_2nd.json', 'r') as f:
    for line in f:
        try:
            data.append(json.loads(line))  # Load each JSON object separately
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")
            continue  # Skip malformed lines

# Convert the list of JSON objects into a pandas DataFrame
df = pd.DataFrame(data)

# Ensure your dataframe has the required columns (instruction, input, output)
df.head(2)

,instruction,label,text
0,Analyze the sentence for any emotional content.,['No emotions'],من يبي موجب مشعر كبير سالب جده مبادل جدة موجب جد
1,List any emotional undertones that are detecta...,['Disgust'],@19rbb ايش ذا السؤال الغبي بالله يعني ايام زما...


In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", # Supports zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, unsloth
    mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}, # ShareGPT style
    map_eos_token = True, # Maps <|im_end|> to </s> instead
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }
pass

Unsloth: Will map <|im_end|> to EOS = </s>.
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


In [ ]:
df.rename(columns={'instruction': 'instruction', 'text': 'input', 'label': 'output'}, inplace=True)

In [ ]:
# Convert the DataFrame to a Hugging Face Dataset
dataset = Dataset.from_pandas(df)

In [ ]:
df.head(2)

,instruction,output,input
0,Analyze the sentence for any emotional content.,['No emotions'],من يبي موجب مشعر كبير سالب جده مبادل جدة موجب جد
1,List any emotional undertones that are detecta...,['Disgust'],@19rbb ايش ذا السؤال الغبي بالله يعني ايام زما...


In [ ]:
len(df)

8000

In [ ]:
# Ensure the tokenizer and EOS_TOKEN are properly defined
EOS_TOKEN = tokenizer.eos_token

# Define the alpaca prompt template
alpaca_prompt = """You are an expert social media text analyzer specializing in identifying emotion content in Arabic contexts. To accurately perform this task, you will pay close attention to the text to classify them into emotion categories. There are a total of 12 categories: neutral, anger, anticipation, disgust, fear, joy, love, optimism, pessimism, sadness, surprise, trust. A text can be classified into either one category or multiple categories of emotion. Your task is to identify all the possible categories. Finally, provide the response in valid JSON format.
### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Properly format the output field as a JSON structure
        if isinstance(output, list):  # Assuming 'output' is a list of emotions
            json_output = {"label": output}  # Convert list to JSON-like dict
        else:
            json_output = {"label": [output]}  # Ensure it's wrapped in a list

        # Format the prompt with instruction, input, and JSON-formatted output
        text = alpaca_prompt.format(instruction, input, json_output) + EOS_TOKEN
        texts.append(text)
        #print(texts)  # You can use this for debugging
    return {"text": texts}

# Optional: Print the formatted dataset to verify
#print(dataset)

In [ ]:
# Apply formatting function
dataset = dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

In [ ]:
dataset.shape[0]

8000

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

Converting train dataset to ChatML (num_proc=2):   0%|          | 0/8000 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=2):   0%|          | 0/8000 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/8000 [00:00<?, ? examples/s]

Truncating train dataset (num_proc=2):   0%|          | 0/8000 [00:00<?, ? examples/s]

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040/3,800,305,664 (1.10% trained)
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: fatemamojibor51 (fatemamojibor51-university-of-michigan) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.347000
2,2.318200
3,2.190300
4,2.021000
5,1.805700
6,1.639300
7,1.291000
8,1.159700
9,1.081600
10,0.982200


In [ ]:
# Test set begins here
import json
FastLanguageModel.for_inference(model)
# Load the test data from the JSON file
test_data = []
with open('/content/sample_data/train_2nd.json', 'r') as f:
    for line in f:
        try:
            test_data.append(json.loads(line))  # Load each JSON object separately
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")
            continue  # Skip malformed lines

# Iterate through the test data and generate outputs
generated_outputs = []
for entry in test_data:
    instruction = entry['instruction']  # Read the instruction
    input_text = entry['text']          # Read the input text

    # Format the prompt for the model
    inputs = tokenizer(
        [
            alpaca_prompt.format(
                instruction,  # instruction
                input_text,   # input text from JSON
                ""            # Leave output blank for generation
            )
        ], return_tensors="pt").to("cuda")

    # Generate the output from the model
    outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)

    # Decode the generated output
    decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # Add the generated output back to the test data
    entry['generated_output'] = decoded_output
    generated_outputs.append(entry)

# Optional: Print the generated outputs to verify
for output in generated_outputs:
    print(f"Instruction: {output['instruction']}")
    print(f"Input: {output['text']}")
    print(f"Generated Output: {output['generated_output']}")
    print()

# Save the results with generated outputs back to a new JSON file
with open('test_results_with_predictions.json', 'w') as outfile:
    json.dump(generated_outputs, outfile, ensure_ascii=False, indent=4)

# Test set ends here

In [ ]:
# Test set begins here
import json

# Load the test data from the JSON file
test_data = []
with open('/content/test_emotion_instruction.json', 'r') as f:
    for line in f:
        try:
            test_data.append(json.loads(line))  # Load each JSON object separately
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")
            continue  # Skip malformed lines

# Iterate through the test data and generate outputs
generated_outputs = []
for entry in test_data:
    instruction = entry['instruction']  # Read the instruction
    input_text = entry['text']          # Read the input text

    # Format the prompt for the model
    inputs = tokenizer(
        [
            alpaca_prompt.format(
                instruction,  # instruction
                input_text,   # input text from JSON
                ""            # Leave output blank for generation
            )
        ], return_tensors="pt").to("cuda")

    # Generate the output from the model
    outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)

    # Decode the generated output
    decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # Strip off unnecessary parts (since we only need the actual "Response")
    response_start = "### Response:\n"
    if response_start in decoded_output:
        decoded_output = decoded_output.split(response_start)[-1].strip()

    # Add only the decoded output to the test data
    entry['generated_output'] = decoded_output
    generated_outputs.append(entry)

# Optional: Print the generated outputs to verify
for output in generated_outputs:
    print(f"Instruction: {output['instruction']}")
    print(f"Input: {output['text']}")
    print(f"Generated Output: {output['generated_output']}")
    print()

# Save the results with generated outputs back to a new JSON file
with open('test_results_with_predictions.json', 'w') as outfile:
    json.dump(generated_outputs, outfile, ensure_ascii=False, indent=4)

# Test set ends here

In [ ]:
from google.colab import drive
drive.mount('/content/drive')